# Task 3: Data Visualization & Dashboarding

> **ApexPlanet Data Analytics Internship**  
> *30-Day Program | Day 14-20*

## What You'll Learn
- Day 14-15: Python Visualization (Matplotlib, Seaborn, Plotly)
- Day 16-18: Power BI / Tableau Basics
- Day 19-20: Interactive Dashboard with 6-8 visuals

---

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

## Step 1: Load the Dataset

We use the same Superstore Sales dataset from Tasks 1 & 2.

In [ ]:
# Load cleaned data from Task 1
df = pd.read_csv('../data/superstore_cleaned.csv')
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'])

print(f"📊 Dataset loaded: {len(df)} rows x {len(df.columns)} columns")
print(f"📅 Date range: {df['Order_Date'].min().date()} to {df['Order_Date'].max().date()}")
print(f"\n📋 Columns: {list(df.columns)}")

## Step 2: Matplotlib Static Charts (Day 14-15)

Matplotlib is great for creating publication-quality static charts.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Matplotlib Visualizations', fontsize=16, fontweight='bold', y=1.02)

# Line chart - Monthly Sales Trend
monthly = df.groupby([df['Order_Date'].dt.to_period('M')])['Sales'].sum()
monthly.index = monthly.index.to_timestamp()
axes[0,0].plot(monthly.index, monthly.values, color='#3498DB', linewidth=2)
axes[0,0].fill_between(monthly.index, monthly.values, alpha=0.3, color='#3498DB')
axes[0,0].set_title('Monthly Sales Trend', fontweight='bold')
axes[0,0].set_ylabel('Sales ($)')

# Bar chart - Sales by Category
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
axes[0,1].bar(cat_sales.index, cat_sales.values, color=['#E74C3C', '#2ECC71', '#9B59B6'])
axes[0,1].set_title('Sales by Category', fontweight='bold')
axes[0,1].set_ylabel('Sales ($)')
for i, v in enumerate(cat_sales.values):
    axes[0,1].text(i, v + 5000, f'${v:,.0f}', ha='center', fontweight='bold')

# Scatter plot - Sales vs Profit
for cat in df['Category'].unique():
    cat_data = df[df['Category'] == cat]
    axes[1,0].scatter(cat_data['Sales'], cat_data['Profit'], label=cat, alpha=0.5, s=20)
axes[1,0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1,0].set_xlabel('Sales ($)')
axes[1,0].set_ylabel('Profit ($)')
axes[1,0].set_title('Sales vs Profit by Category', fontweight='bold')
axes[1,0].legend()

# Histogram - Sales Distribution
axes[1,1].hist(df['Sales'], bins=50, color='#F39C12', edgecolor='black', alpha=0.7)
axes[1,1].axvline(df['Sales'].mean(), color='red', linestyle='--', label=f'Mean: ${df["Sales"].mean():.0f}')
axes[1,1].axvline(df['Sales'].median(), color='green', linestyle='--', label=f'Median: ${df["Sales"].median():.0f}')
axes[1,1].set_title('Sales Distribution', fontweight='bold')
axes[1,1].set_xlabel('Sales ($)')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('../images/task3/matplotlib_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Matplotlib charts saved!")

## Step 3: Seaborn Statistical Visualizations (Day 14-15)

Seaborn is built on top of Matplotlib and makes statistical plots easier.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Seaborn Statistical Visualizations', fontsize=16, fontweight='bold', y=1.02)

# Heatmap - Correlation Matrix
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping_Days']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, square=True, 
            linewidths=1, fmt='.2f', ax=axes[0,0])
axes[0,0].set_title('Correlation Heatmap', fontweight='bold')

# Boxplot - Sales by Category
sns.boxplot(data=df, x='Category', y='Sales', ax=axes[0,1])
axes[0,1].set_title('Sales Distribution by Category (Boxplot)', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=15)

# Scatter with size
sns.scatterplot(data=df, x='Sales', y='Profit', hue='Category', 
                size='Quantity', sizes=(20, 200), alpha=0.6, ax=axes[1,0])
axes[1,0].set_title('Sales vs Profit (Sized by Quantity)', fontweight='bold')
axes[1,0].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Violin plot - Profit by Segment
sns.violinplot(data=df, x='Segment', y='Profit', ax=axes[1,1])
axes[1,1].set_title('Profit Distribution by Segment (Violin)', fontweight='bold')

plt.tight_layout()
plt.savefig('../images/task3/seaborn_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Seaborn charts saved!")

## Step 4: Plotly Interactive Visualizations (Day 14-15)

Plotly creates interactive charts that you can zoom, hover, and explore.

In [ ]:
# Interactive 1: Sales Trend with Hover
monthly_df = df.groupby([df['Order_Date'].dt.to_period('M').dt.to_timestamp()]).agg({
    'Sales': 'sum', 'Profit': 'sum', 'Order_ID': 'count'
}).reset_index()
monthly_df.columns = ['Date', 'Sales', 'Profit', 'Orders']

fig1 = px.line(monthly_df, x='Date', y='Sales', 
               title='Interactive: Monthly Sales Trend',
               labels={'Sales': 'Total Sales ($)', 'Date': 'Month'},
               template='plotly_white')
fig1.update_traces(line_color='#3498DB', line_width=3)
fig1.update_layout(title_font_size=18, title_x=0.5)
fig1.write_html('../images/task3/plotly_sales_trend.html')
fig1.show()
print("✅ Interactive sales trend saved!")

In [ ]:
# Interactive 2: Sunburst Chart
fig2 = px.sunburst(df, path=['Category', 'Sub_Category'], values='Sales',
                   title='Interactive: Sales Hierarchy (Sunburst)',
                   color='Category', template='plotly_white')
fig2.update_layout(title_font_size=18, title_x=0.5)
fig2.write_html('../images/task3/plotly_sunburst.html')
fig2.show()
print("✅ Sunburst chart saved!")

In [ ]:
# Interactive 3: Treemap
fig3 = px.treemap(df, path=['Region', 'State'], values='Sales',
                  title='Interactive: Sales by Region & State (Treemap)',
                  color='Profit', color_continuous_scale='RdYlGn',
                  template='plotly_white')
fig3.update_layout(title_font_size=18, title_x=0.5)
fig3.write_html('../images/task3/plotly_treemap.html')
fig3.show()
print("✅ Treemap saved!")

In [ ]:
# Interactive 4: Animated Scatter
yearly_state = df.groupby(['Order_Year', 'State']).agg({
    'Sales': 'sum', 'Profit': 'sum', 'Order_ID': 'count'
}).reset_index()
yearly_state.columns = ['Year', 'State', 'Sales', 'Profit', 'Orders']

fig4 = px.scatter(yearly_state, x='Sales', y='Profit', size='Orders', 
                    color='State', animation_frame='Year',
                    title='Animated: Sales vs Profit by State Over Time',
                    template='plotly_white', size_max=60)
fig4.update_layout(title_font_size=18, title_x=0.5)
fig4.write_html('../images/task3/plotly_animated_scatter.html')
fig4.show()
print("✅ Animated scatter saved!")

## Step 5: Executive Dashboard (Day 16-18)

Build a professional dashboard with KPI cards and multiple visuals.

In [ ]:
# Calculate KPIs
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = len(df)
unique_customers = df['Customer_Name'].nunique()
avg_order_value = df['Sales'].mean()
profit_margin = (total_profit / total_sales) * 100

print(f"💰 Total Sales: ${total_sales:,.2f}")
print(f"📈 Total Profit: ${total_profit:,.2f}")
print(f"🛒 Total Orders: {total_orders:,}")
print(f"👥 Unique Customers: {unique_customers:,}")
print(f"💵 Avg Order Value: ${avg_order_value:.2f}")
print(f"📊 Profit Margin: {profit_margin:.2f}%")

In [ ]:
# Create Executive Dashboard
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

fig.suptitle('EXECUTIVE DASHBOARD - Superstore Sales Analytics', 
             fontsize=20, fontweight='bold', y=0.98)

# KPI Cards (Top Row)
kpis = [
    ('Total Sales', f'${total_sales/1e6:.2f}M', '#2ECC71'),
    ('Total Profit', f'${total_profit/1e6:.2f}M', '#3498DB'),
    ('Total Orders', f'{total_orders:,}', '#E74C3C')
]

for i, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.text(0.5, 0.65, value, ha='center', va='center', 
            fontsize=24, fontweight='bold', color=color)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=12, color='gray')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, 
                                 edgecolor=color, linewidth=3, linestyle='--'))

# Sales Trend (Middle Left)
ax1 = fig.add_subplot(gs[1, :2])
monthly = df.groupby([df['Order_Date'].dt.to_period('M')])['Sales'].sum()
monthly.index = monthly.index.to_timestamp()
ax1.plot(monthly.index, monthly.values, color='#3498DB', linewidth=2.5, marker='o', markersize=4)
ax1.fill_between(monthly.index, monthly.values, alpha=0.2, color='#3498DB')
ax1.set_title('Sales Trend Over Time', fontweight='bold', fontsize=14)
ax1.set_ylabel('Sales ($)')
ax1.grid(True, alpha=0.3)

# Category Breakdown (Middle Right)
ax2 = fig.add_subplot(gs[1, 2])
cat_sales = df.groupby('Category')['Sales'].sum()
colors = ['#E74C3C', '#2ECC71', '#9B59B6']
wedges, texts, autotexts = ax2.pie(cat_sales, labels=cat_sales.index, autopct='%1.1f%%',
                                     colors=colors, startangle=90)
ax2.set_title('Sales by Category', fontweight='bold', fontsize=14)

# Top 10 Products (Bottom Left)
ax3 = fig.add_subplot(gs[2, 0])
top_products = df.groupby('Product_Name')['Sales'].sum().sort_values(ascending=True).tail(10)
ax3.barh(range(len(top_products)), top_products.values, color='#F39C12')
ax3.set_yticks(range(len(top_products)))
ax3.set_yticklabels([p[:20] + '...' if len(p) > 20 else p for p in top_products.index], fontsize=9)
ax3.set_title('Top 10 Products', fontweight='bold', fontsize=12)
ax3.set_xlabel('Sales ($)')

# Regional Performance (Bottom Middle)
ax4 = fig.add_subplot(gs[2, 1])
reg_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
bars = ax4.bar(reg_sales.index, reg_sales.values, color=['#1ABC9C', '#3498DB', '#9B59B6', '#E67E22'])
ax4.set_title('Sales by Region', fontweight='bold', fontsize=12)
ax4.set_ylabel('Sales ($)')
for bar, val in zip(bars, reg_sales.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000, 
             f'${val:,.0f}', ha='center', fontweight='bold', fontsize=9)

# Segment Analysis (Bottom Right)
ax5 = fig.add_subplot(gs[2, 2])
seg_data = df.groupby('Segment').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
x = np.arange(len(seg_data))
width = 0.35
ax5.bar(x - width/2, seg_data['Sales'], width, label='Sales', color='#3498DB', alpha=0.8)
ax5.bar(x + width/2, seg_data['Profit'], width, label='Profit', color='#2ECC71', alpha=0.8)
ax5.set_xticks(x)
ax5.set_xticklabels(seg_data['Segment'])
ax5.set_title('Sales vs Profit by Segment', fontweight='bold', fontsize=12)
ax5.legend()

plt.savefig('../images/task3/executive_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Executive Dashboard saved!")

## Step 6: Interactive Filter Dashboard (Day 19-20)

A comprehensive dashboard with filter-like panels.

In [ ]:
fig = plt.figure(figsize=(22, 16))
fig.suptitle('INTERACTIVE DASHBOARD - Sales Analytics with Filters', 
             fontsize=22, fontweight='bold', y=0.98)

gs = fig.add_gridspec(4, 4, hspace=0.35, wspace=0.35)

# Row 0: KPI Cards (4 cards)
kpis = [
    ('Total Sales', f'${total_sales/1e6:.2f}M', '#2ECC71'),
    ('Total Profit', f'${total_profit/1e6:.2f}M', '#3498DB'),
    ('Total Orders', f'{total_orders:,}', '#E74C3C'),
    ('Avg Order', f'${avg_order_value:.0f}', '#9B59B6')
]

for i, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.text(0.5, 0.65, value, ha='center', va='center', 
            fontsize=24, fontweight='bold', color=color)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=12, color='gray')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=False, 
                                 edgecolor=color, linewidth=2.5, linestyle='--'))

# Row 1: Sales Trend + Category Breakdown
ax1 = fig.add_subplot(gs[1, :2])
ax1.plot(monthly.index, monthly.values, color='#3498DB', linewidth=2.5, marker='o', markersize=3)
ax1.fill_between(monthly.index, monthly.values, alpha=0.2, color='#3498DB')
ax1.set_title('📈 Sales Trend Over Time', fontweight='bold', fontsize=13)
ax1.set_ylabel('Sales ($)')
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[1, 2:])
wedges, texts, autotexts = ax2.pie(cat_sales, labels=cat_sales.index, autopct='%1.1f%%',
                                     colors=colors, startangle=90, textprops={'fontsize': 10})
ax2.set_title('📊 Category Breakdown', fontweight='bold', fontsize=13)

# Row 2: Top Products + Regional + Ship Mode
ax3 = fig.add_subplot(gs[2, 0])
top_products = df.groupby('Product_Name')['Sales'].sum().sort_values(ascending=True).tail(8)
ax3.barh(range(len(top_products)), top_products.values, color='#F39C12')
ax3.set_yticks(range(len(top_products)))
ax3.set_yticklabels([p[:18] + '..' if len(p) > 18 else p for p in top_products.index], fontsize=8)
ax3.set_title('🏆 Top 8 Products', fontweight='bold', fontsize=12)
ax3.set_xlabel('Sales ($)')

ax4 = fig.add_subplot(gs[2, 1:3])
reg_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
reg_profit = df.groupby('Region')['Profit'].sum()
x = np.arange(len(reg_sales))
width = 0.35
bars1 = ax4.bar(x - width/2, reg_sales.values, width, label='Sales', color='#3498DB', alpha=0.8)
bars2 = ax4.bar(x + width/2, reg_profit.values, width, label='Profit', color='#2ECC71', alpha=0.8)
ax4.set_xticks(x)
ax4.set_xticklabels(reg_sales.index)
ax4.set_title('🌍 Regional Performance', fontweight='bold', fontsize=13)
ax4.legend()

ax5 = fig.add_subplot(gs[2, 3])
ship_data = df.groupby('Ship_Mode')['Sales'].sum().sort_values(ascending=False)
ax5.bar(range(len(ship_data)), ship_data.values, color=['#1ABC9C', '#27AE60', '#2980B9', '#8E44AD'])
ax5.set_xticks(range(len(ship_data)))
ax5.set_xticklabels(ship_data.index, rotation=45, ha='right', fontsize=9)
ax5.set_title('🚚 Ship Mode Sales', fontweight='bold', fontsize=12)

# Row 3: Monthly Orders + Discount Impact + Quarterly Trend
ax6 = fig.add_subplot(gs[3, 0])
monthly_orders = df.groupby([df['Order_Date'].dt.to_period('M')])['Order_ID'].count()
monthly_orders.index = monthly_orders.index.to_timestamp()
ax6.bar(monthly_orders.index, monthly_orders.values, color='#E67E22', alpha=0.7)
ax6.set_title('📦 Monthly Order Volume', fontweight='bold', fontsize=12)
ax6.set_ylabel('Orders')

ax7 = fig.add_subplot(gs[3, 1:3])
discount_bins = pd.cut(df['Discount'], bins=[0, 0.01, 0.1, 0.2, 0.3], 
                        labels=['No Discount', 'Low (1-10%)', 'Medium (10-20%)', 'High (>20%)'])
discount_impact = df.groupby(discount_bins)['Profit'].mean()
colors_disc = ['#2ECC71', '#F1C40F', '#E67E22', '#E74C3C']
ax7.bar(range(len(discount_impact)), discount_impact.values, color=colors_disc)
ax7.set_xticks(range(len(discount_impact)))
ax7.set_xticklabels(discount_impact.index, rotation=15, fontsize=10)
ax7.set_title('💰 Discount Impact on Profit', fontweight='bold', fontsize=13)
ax7.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
for i, v in enumerate(discount_impact.values):
    ax7.text(i, v + 5, f'${v:.0f}', ha='center', fontweight='bold')

ax8 = fig.add_subplot(gs[3, 3])
quarterly = df.groupby(['Order_Year', 'Order_Quarter'])['Sales'].sum().reset_index()
quarterly['Label'] = quarterly['Order_Year'].astype(str) + '-Q' + quarterly['Order_Quarter'].astype(str)
ax8.plot(range(len(quarterly)), quarterly['Sales'], marker='D', color='#9B59B6', linewidth=2)
ax8.fill_between(range(len(quarterly)), quarterly['Sales'], alpha=0.2, color='#9B59B6')
ax8.set_title('📅 Quarterly Trend', fontweight='bold', fontsize=12)
ax8.set_xticks(range(0, len(quarterly), 2))
ax8.set_xticklabels(quarterly['Label'].iloc[::2], rotation=45, ha='right', fontsize=8)

plt.savefig('../images/task3/interactive_filter_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Interactive Filter Dashboard saved!")

## Step 7: Power BI / Tableau Data Preparation (Day 16-18)

Create clean datasets with calculated fields for dashboard tools.

In [ ]:
# Create a clean dataset for Power BI / Tableau
df_powerbi = df.copy()

# Add calculated columns
df_powerbi['Profit_Margin'] = (df_powerbi['Profit'] / df_powerbi['Sales']) * 100
df_powerbi['Year'] = df_powerbi['Order_Date'].dt.year
df_powerbi['Month'] = df_powerbi['Order_Date'].dt.month
df_powerbi['Month_Name'] = df_powerbi['Order_Date'].dt.month_name()
df_powerbi['Quarter'] = df_powerbi['Order_Date'].dt.quarter
df_powerbi['Day_of_Week'] = df_powerbi['Order_Date'].dt.day_name()
df_powerbi['Is_Weekend'] = df_powerbi['Order_Date'].dt.dayofweek >= 5
df_powerbi['Sales_Category'] = pd.cut(df_powerbi['Sales'], 
                                       bins=[0, 100, 500, 1000, 5000, float('inf')],
                                       labels=['<$100', '$100-500', '$500-1K', '$1K-5K', '>$5K'])
df_powerbi['Profit_Status'] = df_powerbi['Profit'].apply(lambda x: 'Profitable' if x > 0 else 'Loss')
df_powerbi['Discount_Category'] = pd.cut(df_powerbi['Discount'],
                                          bins=[-0.1, 0, 0.1, 0.2, 0.3, 1],
                                          labels=['No Discount', 'Low', 'Medium', 'High', 'Very High'])

# Save as CSV for Power BI / Tableau
df_powerbi.to_csv('../dashboards/superstore_for_powerbi.csv', index=False)
print("✅ Power BI / Tableau dataset saved!")
print(f"   Columns: {len(df_powerbi.columns)} (including calculated fields)")

# Create daily summary
summary_by_date = df.groupby([df['Order_Date'].dt.date]).agg({
    'Sales': 'sum', 'Profit': 'sum', 'Quantity': 'sum',
    'Discount': 'mean', 'Order_ID': 'count'
}).reset_index()
summary_by_date.columns = ['Date', 'Total_Sales', 'Total_Profit', 'Total_Quantity', 'Avg_Discount', 'Order_Count']
summary_by_date['Profit_Margin'] = (summary_by_date['Total_Profit'] / summary_by_date['Total_Sales']) * 100
summary_by_date.to_csv('../dashboards/daily_summary.csv', index=False)
print("✅ Daily summary saved!")

# Create KPI summary
kpi_summary = pd.DataFrame({
    'KPI': ['Total Sales', 'Total Profit', 'Total Orders', 'Unique Customers', 
            'Avg Order Value', 'Profit Margin %', 'Avg Discount %'],
    'Value': [total_sales, total_profit, total_orders, unique_customers,
              avg_order_value, profit_margin, df['Discount'].mean() * 100],
    'Format': ['Currency', 'Currency', 'Number', 'Number', 'Currency', 'Percentage', 'Percentage']
})
kpi_summary.to_csv('../dashboards/kpi_summary.csv', index=False)
print("✅ KPI summary saved!")

print("\n📊 All dashboard data files ready for Power BI / Tableau!")

## Summary

This notebook demonstrated:
- ✅ Matplotlib static charts (line, bar, scatter, histogram)
- ✅ Seaborn statistical plots (heatmap, boxplot, violin)
- ✅ Plotly interactive visualizations (line, sunburst, treemap, animated)
- ✅ Executive Dashboard with KPI cards and multiple visuals
- ✅ Interactive Filter Dashboard with 8 panels
- ✅ Power BI / Tableau data preparation

**All charts saved to:** `../images/task3/`
**Dashboard data saved to:** `../dashboards/`